# Health Appointment Scheduling Chatbot

### Interactive chatbot for scheduling medical visits using slot-filling logic

This notebook demonstrates a Python-based chatbot that helps users book health appointments. It collects structured information like doctor type, preferred time, and location through a friendly conversation.

## OpenAI Logic

In [2]:
from dotenv import load_dotenv
import os

# with open(".env", "w") as f:
#     f.write("OPENAI_API_KEY=YOUR_OPENAI_KEY_GOES_HERE\n")

load_dotenv()  # Loads variables from .env

api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:3], "*****")  # just to confirm it's loaded

sk- *****


In [5]:
import os
import json
from typing import Dict, List, Any, Optional
from openai import OpenAI

class HealthAppointmentOpenAIChatbot:
    def __init__(self, api_key: str = None):
        if api_key:
            self.client = OpenAI(api_key=api_key)
        else:
            api_key = os.getenv("OPENAI_API_KEY")
            if not api_key:
                raise ValueError("API key must be provided or set as OPENAI_API_KEY environment variable")
            self.client = OpenAI(api_key=api_key)

        self.conversation_history = []

        self.required_slots = [
            "doctor_type", "preferred_day", "preferred_time", "location"
        ]

        self.optional_slots = [
            "insurance", "urgency", "preferred_gender", "language", "symptoms"
        ]

        self.collected_info = {slot: None for slot in self.required_slots + self.optional_slots}
        self.state = "greeting"

    def add_message_to_history(self, role: str, content: str):
        self.conversation_history.append({"role": role, "content": content})

    def extract_appointment_info(self, user_message: str) -> Dict[str, Any]:
        messages = [
            {"role": "system", "content": "You are a healthcare assistant helping extract medical appointment details. Extract ONLY what is explicitly stated. Format the output as JSON."}
        ]
        messages.extend(self.conversation_history[-5:])
        messages.append({"role": "user", "content": f"Extract appointment preferences from this message: '{user_message}'. Return ONLY JSON."})

        tools = [
            {
                "type": "function",
                "function": {
                    "name": "extract_appointment_preferences",
                    "description": "Extract medical appointment preferences from user input",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "doctor_type": {"type": "string"},
                            "preferred_day": {"type": "string"},
                            "preferred_time": {"type": "string"},
                            "location": {"type": "string"},
                            "insurance": {"type": "string"},
                            "urgency": {"type": "string"},
                            "preferred_gender": {"type": "string"},
                            "language": {"type": "string"},
                            "symptoms": {"type": "string"}
                        },
                        "required": []
                    }
                }
            }
        ]

        try:
            response = self.client.chat.completions.create(
                model="gpt-4-turbo",
                messages=messages,
                tools=tools,
                tool_choice={"type": "function", "function": {"name": "extract_appointment_preferences"}}
            )
            tool_calls = response.choices[0].message.tool_calls
            if tool_calls and tool_calls[0].function.name == "extract_appointment_preferences":
                return json.loads(tool_calls[0].function.arguments)
            return {}
        except Exception as e:
            print(f"Error calling OpenAI API: {e}")
            return {}

    def update_collected_info(self, extracted_info: Dict[str, Any]):
        for slot, value in extracted_info.items():
            if value:
                self.collected_info[slot] = value

    def get_missing_required_slots(self) -> List[str]:
        return [slot for slot in self.required_slots if not self.collected_info[slot]]

    def generate_next_question(self) -> str:
        messages = [
            {"role": "system", "content": "You are a friendly healthcare assistant. Ask one missing piece of appointment info conversationally and briefly."},
            {"role": "user", "content": f"Collected: {json.dumps(self.collected_info)}\nMissing: {', '.join(self.get_missing_required_slots())}"}
        ]

        try:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                max_tokens=100
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error generating question: {e}")
            return "Can you tell me more about your appointment preferences?"

    def generate_confirmation_message(self) -> str:
        messages = [
            {"role": "system", "content": "Summarize patient appointment info and ask for confirmation."},
            {"role": "user", "content": f"Summary: {json.dumps({k: v for k, v in self.collected_info.items() if v})}"}
        ]

        try:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                max_tokens=150
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error generating confirmation: {e}")
            return f"Does this look right? {json.dumps(self.get_structured_data())}"

    def get_structured_data(self) -> Dict[str, Any]:
        return {k: v for k, v in self.collected_info.items() if v}

    def process_message(self, user_message: str) -> str:
        self.add_message_to_history("user", user_message)

        if self.state == "confirming":
            if any(w in user_message.lower() for w in ["yes", "correct", "sure", "ok"]):
                self.state = "completed"
                result = f" Appointment confirmed with: {json.dumps(self.get_structured_data())}"
                self.add_message_to_history("assistant", result)
                return result
            elif any(w in user_message.lower() for w in ["no", "wrong", "change"]):
                self.state = "collecting"
                return "Okay! What would you like to change?"

        if self.state == "greeting":
            self.state = "collecting"
            greeting = "Hi! I'm your health appointment assistant. Let's schedule your visit."
            self.add_message_to_history("assistant", greeting)
            return greeting

        extracted_info = self.extract_appointment_info(user_message)
        self.update_collected_info(extracted_info)

        if not self.get_missing_required_slots():
            self.state = "confirming"
            confirmation = self.generate_confirmation_message()
            self.add_message_to_history("assistant", confirmation)
            return confirmation
        else:
            next_q = self.generate_next_question()
            self.add_message_to_history("assistant", next_q)
            return next_q

def run_chatbot_interactive():
    api_key = input("Enter your OpenAI API key (or leave blank if using environment variable): ")
    if not api_key:
        api_key = None

    chatbot = HealthAppointmentOpenAIChatbot(api_key=api_key)
    print("Bot:", chatbot.process_message("start"))

    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            break
        response = chatbot.process_message(user_input)
        print("Bot:", response)
        if chatbot.state == "completed":
            print("\n Final data:", chatbot.get_structured_data())
            break

run_chatbot_interactive()

Enter your OpenAI API key (or leave blank if using environment variable):  


Bot: Hi! I'm your health appointment assistant. Let's schedule your visit.


You:  Hi, i need a dentist


Bot: Oh, before we finalize your appointment, may I ask when you would prefer to come in and at what time? Also, do you have a preferred location for the appointment?


You:  Friday evening


Bot: Great, I have all the details for your appointment except for the location. Could you please provide me with the preferred location for your dental appointment?


You:  Anywhere in Poznan


Bot: Would you like to confirm your appointment with a dentist on Friday evening at any location in Poznan?


You:  yes


Bot:  Appointment confirmed with: {"doctor_type": "dentist", "preferred_day": "Friday", "preferred_time": "evening", "location": "Anywhere in Poznan"}

 Final data: {'doctor_type': 'dentist', 'preferred_day': 'Friday', 'preferred_time': 'evening', 'location': 'Anywhere in Poznan'}


## Langchain Logic

In [ ]:
## Imports and Setup
import os
import json
from typing import Dict, List, Any, Optional, Union
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.schema import OutputParserException
from langchain_core.output_parsers import StrOutputParser

## Define Appointment Requirements (Pydantic Schema)

class AppointmentPreferences(BaseModel):
    doctor_type: Optional[str] = Field(None, description="Type of doctor needed (dentist, dermatologist, etc.)")
    preferred_day: Optional[str] = Field(None, description="Preferred day of the week")
    preferred_time: Optional[str] = Field(None, description="Preferred time or time window")
    location: Optional[str] = Field(None, description="City or clinic for the appointment")
    insurance: Optional[str] = Field(None, description="Insurance provider name")
    urgency: Optional[str] = Field(None, description="Whether this is urgent or not")


## LangChain Chatbot Class

class HealthAppointmentLangchainChatbot:
    def __init__(self, api_key: str = None):
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
        elif not os.getenv("OPENAI_API_KEY"):
            raise ValueError("API key must be provided or set as OPENAI_API_KEY")

        self.extraction_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.1)
        self.response_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
        self.parser = PydanticOutputParser(pydantic_object=AppointmentPreferences)
        self.memory = ConversationBufferMemory(return_messages=True)
        self.required_slots = ["doctor_type", "preferred_day", "preferred_time", "location"]
        self.preferences = AppointmentPreferences()
        self.state = "collecting"
        self.first = True


## Extract Appointment Info
    def extract_preferences(self, user_message: str) -> AppointmentPreferences:
        prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""
                You are a helpful assistant. Extract appointment booking preferences from the user's message.
                Do NOT guess. Only extract what's explicitly stated.
            """),
            MessagesPlaceholder(variable_name="history"),
            HumanMessage(content=f"Extract info from: '{user_message}'"),
            SystemMessage(content=self.parser.get_format_instructions())
        ])

        chain = prompt | self.extraction_llm | StrOutputParser()

        try:
            result = chain.invoke({"history": self.memory.load_memory_variables({})["history"]})
            try:
                data = json.loads(result)
                return AppointmentPreferences(**data)
            except:
                return self.parser.parse(result)
        except Exception as e:
            print("Extraction error:", e)
            return AppointmentPreferences()


## Update Collected Information

    def update_preferences(self, new_info: AppointmentPreferences):
        current = self.preferences.model_dump()
        extracted = {k: v for k, v in new_info.model_dump().items() if v}
        current.update(extracted)
        self.preferences = AppointmentPreferences(**current)


## Ask Follow-Up Questions

    def get_missing_slots(self) -> List[str]:
        return [s for s in self.required_slots if not getattr(self.preferences, s)]

    def ask_next_question(self) -> str:
        missing = self.get_missing_slots()
        filled = {k: v for k, v in self.preferences.model_dump().items() if v}

        if not missing:
            return self.confirmation_prompt()

        prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""
                You are a healthcare assistant. Ask the user for exactly one missing field, based on their data.
                Don't greet again, and be specific.
            """),
            SystemMessage(content=f"User so far: {json.dumps(filled)}\nMissing: {', '.join(missing)}")
        ])

        chain = prompt | self.response_llm | StrOutputParser()
        try:
            return chain.invoke({})
        except:
            return f"Can you tell me your {missing[0].replace('_', ' ')}?"


## Confirm Information

    def confirmation_prompt(self) -> str:
        filled = {k: v for k, v in self.preferences.model_dump().items() if v}

        prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""
                Summarize the user's appointment preferences and ask for confirmation.
            """),
            SystemMessage(content=f"Collected info: {json.dumps(filled)}")
        ])

        chain = prompt | self.response_llm | StrOutputParser()
        try:
            return chain.invoke({})
        except:
            text = ", ".join([f"{k}: {v}" for k, v in filled.items()])
            return f"You're booking an appointment with: {text}. Is this correct?"


##  Main Chat Flow
    def process(self, user_message: str) -> str:
        if self.first and user_message.lower() in ["hi", "hello", "start"]:
            self.first = False
            self.memory.chat_memory.add_user_message(user_message)
            greeting = "Hi! What kind of doctor do you need?"
            self.memory.chat_memory.add_ai_message(greeting)
            return greeting

        self.memory.chat_memory.add_user_message(user_message)
        self.first = False

        if self.state == "confirming":
            if any(w in user_message.lower() for w in ["yes", "correct", "looks good"]):
                self.state = "done"
                summary = f"Info saved: {json.dumps(self.get_structured_data(), indent=2)}"
                self.memory.chat_memory.add_ai_message(summary)
                return summary
            elif any(w in user_message.lower() for w in ["no", "wrong", "change"]):
                self.state = "collecting"
                msg = "Got it. What would you like to update?"
                self.memory.chat_memory.add_ai_message(msg)
                return msg

        new_info = self.extract_preferences(user_message)
        self.update_preferences(new_info)

        if not self.get_missing_slots():
            self.state = "confirming"
            confirm = self.confirmation_prompt()
            self.memory.chat_memory.add_ai_message(confirm)
            return confirm

        question = self.ask_next_question()
        self.memory.chat_memory.add_ai_message(question)
        return question

    def get_structured_data(self) -> Dict[str, Any]:
        return {k: v for k, v in self.preferences.model_dump().items() if v}


## Run It Interactively

def run_health_langchain_chat():
    api_key = input("Enter your OpenAI API key (or press Enter to use ENV variable): ")
    if not api_key:
        api_key = None

    bot = HealthAppointmentLangchainChatbot(api_key)
    print("🩺 Chatbot ready. Type 'exit' to quit.")
    print("Bot:", bot.process("start"))

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            break
        response = bot.process(user_input)
        print("Bot:", response)

if __name__ == "__main__":
    run_health_langchain_chat()

Enter your OpenAI API key (or press Enter to use ENV variable):  


🩺 Chatbot ready. Type 'exit' to quit.
Bot: Hi! What kind of doctor do you need?


You:  dentist


Bot: What is your preferred day for the appointment?


You:  friday


...

Bot: Where would you prefer to have your appointment with the dentist?


You:  Poznan 


Bot: You have requested to book an appointment with a dentist in Poznan on Friday evening. Is this correct?


You:  yes


Bot: Info saved: {
  "doctor_type": "dentist",
  "preferred_day": "friday",
  "preferred_time": "evening",
  "location": "Poznan"
}


## Regex-based Logic

In [7]:
import re
import json
from typing import Dict, Any, Optional
from datetime import datetime

class HealthAppointmentChatbot:
    def __init__(self):
        self.slots = {
            "doctor_type": {
                "question": "What kind of doctor do you need? (e.g., dentist, cardiologist, dermatologist)",
                "value": None,
                "required": True,
                "validation": lambda x: x.lower() in [
                    "dentist", "cardiologist", "dermatologist", "pediatrician", "therapist", "gp", "surgeon"
                ]
            },
            "preferred_day": {
                "question": "On which day would you like the appointment?",
                "value": None,
                "required": True,
                "validation": lambda x: x.lower() in [
                    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday"
                ]
            },
            "preferred_time": {
                "question": "What time of the day works best? (e.g., morning, afternoon, 10AM)",
                "value": None,
                "required": True,
                "validation": lambda x: bool(re.search(r'\d{1,2}(:\d{2})?\s*(AM|PM|am|pm)?|morning|afternoon|evening', x))
            },
            "location": {
                "question": "Which city or clinic do you prefer?",
                "value": None,
                "required": True,
                "validation": lambda x: len(x.strip()) > 2
            },
            "insurance": {
                "question": "Do you have a specific insurance provider?",
                "value": None,
                "required": False,
                "validation": lambda x: True
            },
            "urgency": {
                "question": "Is this an urgent appointment? (yes/no)",
                "value": None,
                "required": False,
                "validation": lambda x: x.lower() in ["yes", "no"]
            }
        }

        self.current_slot = None
        self.greeting_shown = False
        self.confirmation_mode = False

    # Slot Management Logic
    def get_incomplete_required_slot(self) -> Optional[str]:
        for slot_name, slot_info in self.slots.items():
            if slot_info["required"] and slot_info["value"] is None:
                return slot_name
        return None

    def all_required_slots_filled(self) -> bool:
        return self.get_incomplete_required_slot() is None

    def process_slot_value(self, slot_name: str, value: str) -> bool:
        if self.slots[slot_name]["validation"](value):
            self.slots[slot_name]["value"] = value
            return True
        return False

    def get_confirmation_message(self) -> str:
        info = [f"{k.replace('_', ' ')}: {v['value']}" for k, v in self.slots.items() if v["value"]]
        return "Here’s what I have for your appointment:\n" + "\n".join(info) + "\nIs this correct? (yes/no)"

    def get_structured_data(self) -> Dict[str, Any]:
        return {k: v["value"] for k, v in self.slots.items() if v["value"]}

    # Conversation Engine
    def get_next_question(self) -> str:
        if not self.greeting_shown:
            self.greeting_shown = True
            return "Hello! I'm your health appointment assistant. Let's get you scheduled."

        if self.confirmation_mode:
            return self.get_confirmation_message()

        next_slot = self.get_incomplete_required_slot()
        if next_slot:
            self.current_slot = next_slot
            return self.slots[next_slot]["question"]

        for slot_name, slot_info in self.slots.items():
            if not slot_info["required"] and slot_info["value"] is None:
                self.current_slot = slot_name
                return slot_info["question"]

        self.confirmation_mode = True
        return self.get_confirmation_message()

    def process_message(self, user_message: str) -> str:
        if self.confirmation_mode:
            if "yes" in user_message.lower():
                return f"✅ Appointment request saved: {json.dumps(self.get_structured_data())}"
            elif "no" in user_message.lower():
                self.confirmation_mode = False
                return "Okay, let's update your information. What would you like to change?"

        if self.current_slot and self.slots[self.current_slot]["value"] is None:
            if self.process_slot_value(self.current_slot, user_message):
                pass

        if self.all_required_slots_filled() and not self.confirmation_mode:
            self.confirmation_mode = True

        return self.get_next_question()

# Try It Interactively
def run_health_chatbot():
    chatbot = HealthAppointmentChatbot()
    print("Bot:", chatbot.get_next_question())

    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            break

        response = chatbot.process_message(user_input)
        print("Bot:", response)

        if "✅ Appointment request saved:" in response:
            break

run_health_chatbot()

Bot: Hello! I'm your health appointment assistant. Let's get you scheduled.


You:  Hi


Bot: What kind of doctor do you need? (e.g., dentist, cardiologist, dermatologist)


You:  dentist


Bot: On which day would you like the appointment?


...

Bot: Which city or clinic do you prefer?


You:  Poznan


Bot: Here’s what I have for your appointment:
doctor type: dentist
preferred day: friday
preferred time: evening
location: Poznan
Is this correct? (yes/no)


You:  yes


Bot: ✅ Appointment request saved: {"doctor_type": "dentist", "preferred_day": "friday", "preferred_time": "evening", "location": "Poznan"}
